# Ejercicio Integrador — Deep Neural Networks

En esta actividad construirás, entrenarás y compararás varias redes profundas.

Aplicarás:

- arquitectura de una DNN;
- número de parámetros;
- training loop;
- training y validation performance;
- overfitting;
- Dropout;
- weight decay;
- evaluación del modelo.

Trabajaremos con el dataset **Breast Cancer Wisconsin** incluido en Scikit-learn.

Este dataset contiene mediciones numéricas extraídas de imágenes digitalizadas de muestras celulares.

Nuestro objetivo educativo será clasificar dos categorías usando esas mediciones.

## Instrucciones

Intenta cada sección primero.

Abre la pista si la necesitas.

Consulta la solución solamente después de haber intentado resolver el problema.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report

import torch
import torch.nn as nn


# Parte 1 — Explora los datos

Carga el dataset y determina:

- número de samples;
- número de features;
- número de clases.

### Tu código


In [ ]:
# data = ...
# X = ...
# y = ...



<details>
<summary><strong>Pista</strong></summary>

Usa:

```python
data = load_breast_cancer()
```

</details>

<details>
<summary><strong>Mostrar solución</strong></summary>

```python
data = load_breast_cancer()

X = data.data
y = data.target

print("X.shape =", X.shape)
print("y.shape =", y.shape)
print("Classes =", data.target_names)
```

</details>


# Parte 2 — Training, validation y test

Divide los datos aproximadamente en:

```text
70% training
15% validation
15% test
```

Usa `random_state=42` y `stratify`.


In [ ]:
# Tu código aquí



<details>
<summary><strong>Pista</strong></summary>

Primero separa training y temporal:

```python
X_train, X_temp, y_train, y_temp = train_test_split(...)
```

Después divide `X_temp` en validation y test.

</details>

<details>
<summary><strong>Mostrar solución</strong></summary>

```python
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)
```

</details>


# Parte 3 — Estandarización

Estandariza correctamente usando solamente el training set para ajustar el scaler.


In [ ]:
# Tu código aquí



<details>
<summary><strong>Mostrar solución</strong></summary>

```python
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)
```

</details>


# Parte 4 — Convierte a tensors

Para este problema binario usaremos:

```text
features → float32
labels   → float32
```

y una sola neurona de salida.


In [ ]:
# Tu código aquí



<details>
<summary><strong>Mostrar solución</strong></summary>

```python
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)

y_train_t = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_val_t = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)
y_test_t = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)
```

</details>


# Parte 5 — Diseña una DNN

El dataset tiene 30 features.

Construye:

```text
30 → 64 → 32 → 16 → 1
```

Usa ReLU entre hidden layers.

No apliques Sigmoid dentro del modelo porque utilizaremos `BCEWithLogitsLoss`.


In [ ]:
# model = ...



<details>
<summary><strong>Pista</strong></summary>

La salida final debe ser:

```python
nn.Linear(16, 1)
```

</details>

<details>
<summary><strong>Mostrar solución</strong></summary>

```python
model = nn.Sequential(
    nn.Linear(30, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
)
```

</details>


# Parte 6 — Cuenta los parámetros

Calcula el total manualmente y luego verifica con PyTorch.

### Tu código


In [ ]:
# total_params = ...



<details>
<summary><strong>Pista</strong></summary>

Calcula:

```text
30 → 64
64 → 32
32 → 16
16 → 1
```

</details>

<details>
<summary><strong>Mostrar solución</strong></summary>

```python
total_params = sum(p.numel() for p in model.parameters())
print(total_params)
```

El cálculo manual debe sumar weights y biases de cada capa.

</details>


# Parte 7 — Loss y optimizer

Usa:

```python
nn.BCEWithLogitsLoss()
```

y:

```python
torch.optim.Adam
```

con:

```text
learning rate = 0.001
```


In [ ]:
# criterion = ...
# optimizer = ...



<details>
<summary><strong>Mostrar solución</strong></summary>

```python
criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)
```

</details>


# Parte 8 — Training loop

Entrena durante 300 epochs.

Guarda:

```python
train_history
val_history
```


In [ ]:
epochs = 300

train_history = []
val_history = []

# Completa el training loop



<details>
<summary><strong>Pista</strong></summary>

Durante training:

```python
model.train()
```

Durante validation:

```python
model.eval()

with torch.no_grad():
    ...
```

</details>

<details>
<summary><strong>Mostrar solución</strong></summary>

```python
for epoch in range(epochs):

    model.train()

    logits = model(X_train_t)
    train_loss = criterion(logits, y_train_t)

    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()

    model.eval()

    with torch.no_grad():
        val_logits = model(X_val_t)
        val_loss = criterion(val_logits, y_val_t)

    train_history.append(train_loss.item())
    val_history.append(val_loss.item())
```

</details>


# Parte 9 — Visualiza training y validation loss


In [ ]:
# Tu código aquí



<details>
<summary><strong>Mostrar solución</strong></summary>

```python
plt.figure(figsize=(8, 5))
plt.plot(train_history, label="Training")
plt.plot(val_history, label="Validation")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")
plt.legend()
plt.show()
```

</details>


# Parte 10 — Evalúa accuracy

Completa una función para clasificación binaria.

Recuerda:

```python
probabilities = torch.sigmoid(logits)
```

y:

```text
probability >= 0.5 → class 1
```


In [ ]:
def binary_accuracy(model, X, y):
    # completa
    pass


<details>
<summary><strong>Mostrar solución</strong></summary>

```python
def binary_accuracy(model, X, y):

    model.eval()

    with torch.no_grad():
        logits = model(X)
        probabilities = torch.sigmoid(logits)
        predictions = (probabilities >= 0.5).float()

    return (predictions == y).float().mean().item()
```

</details>


In [ ]:
# Después de completar la función:

# print("Train:", binary_accuracy(model, X_train_t, y_train_t))
# print("Val:  ", binary_accuracy(model, X_val_t, y_val_t))
# print("Test: ", binary_accuracy(model, X_test_t, y_test_t))


# Parte 11 — Añade regularización

Construye ahora:

```text
30 → 64 → Dropout → 32 → Dropout → 16 → 1
```

con:

```text
Dropout = 0.30
```

y añade:

```text
weight_decay = 1e-4
```

al optimizer.


In [ ]:
# regularized_model = ...



<details>
<summary><strong>Pista</strong></summary>

Después de cada ReLU puedes añadir:

```python
nn.Dropout(0.30)
```

</details>

<details>
<summary><strong>Mostrar solución</strong></summary>

```python
regularized_model = nn.Sequential(
    nn.Linear(30, 64),
    nn.ReLU(),
    nn.Dropout(0.30),

    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Dropout(0.30),

    nn.Linear(32, 16),
    nn.ReLU(),

    nn.Linear(16, 1)
)

optimizer = torch.optim.Adam(
    regularized_model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)
```

</details>


# Parte 12 — Compara modelos

Entrena también el modelo regularizado.

Luego compara:

| Métrica | DNN | DNN regularizada |
|---|---:|---:|
| Train accuracy | | |
| Validation accuracy | | |
| Test accuracy | | |
| Parámetros | | |

### Pregunta

¿Regularización necesariamente aumenta training accuracy?

<details>
<summary><strong>Mostrar solución</strong></summary>

No.

Regularización puede incluso reducir ligeramente training accuracy.

El objetivo es mejorar o estabilizar **generalization**, no maximizar el desempeño en training a cualquier costo.

</details>


# Parte 13 — Confusion Matrix

Evalúa el modelo final en el test set.

### Tu código


In [ ]:
# Tu código aquí



<details>
<summary><strong>Pista</strong></summary>

Convierte logits usando:

```python
torch.sigmoid(...)
```

y después aplica threshold 0.5.

</details>

<details>
<summary><strong>Mostrar solución</strong></summary>

```python
regularized_model.eval()

with torch.no_grad():

    logits = regularized_model(X_test_t)
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities >= 0.5).int().view(-1)

true_labels = y_test_t.int().view(-1)

cm = confusion_matrix(
    true_labels.numpy(),
    predictions.numpy()
)

print(cm)
```

</details>


# Desafío final

Experimenta con:

```text
Dropout = 0.10
Dropout = 0.30
Dropout = 0.50
```

y con:

```text
weight_decay = 0
weight_decay = 1e-4
weight_decay = 1e-3
```

Registra tus resultados.

### Preguntas

1. ¿Cuál modelo obtiene menor validation loss?
2. ¿Cuál obtiene mejor test accuracy?
3. ¿Cuál muestra mayor diferencia entre train y validation?
4. ¿Más regularización siempre mejora el modelo?

<details>
<summary><strong>Mostrar interpretación conceptual</strong></summary>

No existe un valor universalmente óptimo.

Muy poca regularización puede permitir overfitting.

Demasiada regularización puede producir **underfitting**.

El objetivo es encontrar un balance entre:

```text
capacidad suficiente
        +
buena generalización
```

</details>


# Checklist final

Al terminar debes poder explicar:

- qué hace profunda a una red;
- por qué una DNN tiene más capacidad;
- cómo calcular sus parámetros;
- qué es overfitting;
- diferencia entre train, validation y test;
- qué hace Dropout;
- qué hace weight decay;
- por qué usamos `model.train()` y `model.eval()`;
- por qué más capas no garantizan mejor desempeño.

El próximo gran paso será trabajar con datos que tienen **estructura espacial**:

# Convolutional Neural Networks


## Recursos

- [PyTorch — Dropout](https://docs.pytorch.org/docs/stable/generated/torch.nn.Dropout.html)
- [PyTorch — BCEWithLogitsLoss](https://docs.pytorch.org/docs/stable/generated/torch.nn.BCEWithLogitsLoss.html)
- [Scikit-learn — Breast Cancer Dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html)
- [Deep Learning Book — Regularization](https://www.deeplearningbook.org/contents/regularization.html)
